## **도구(툴) 활용**

> **한눈에 보기**
> - Tool Schema 작성 원칙, 빌트인 툴(`web_search`), MCP 툴
> - 앞부분(함수 도구와 에이전트 루프)은 [ex1012/exam2.ipynb](../ex1012/exam2.ipynb)
> - 정리 문서: [빌트인툴과MCP.md](빌트인툴과MCP.md)

> 한화내일아카데미 10월 13일 수업 실습 노트북입니다. 강사님 자료 `exam2.ipynb`의 뒷부분입니다.
> - 앞부분(도구가 필요한 이유, 직접 만든 함수 도구, 에이전트 루프, 날씨 도구)은 10월 12일에 실습해서 [`../ex1012/exam2.ipynb`](../ex1012/exam2.ipynb)에 정리했습니다. 여기에는 이날 새로 배운 **Tool Schema 작성 원칙, 빌트인 툴(web_search), MCP 툴**만 남겼습니다.
> - 강의 슬라이드 이미지는 저장소에 넣지 않고 자리만 표시했습니다.
> - **📝 정리** 셀은 실행 결과를 보고 직접 정리한 내용입니다.
> - API 키는 코드에 쓰지 않고 환경변수 `OPENAI_API_KEY`에서 읽습니다(`OpenAI()`가 자동으로 읽음).

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

**1단계: 개발자 ➡ LLM (질문과 메뉴판 전달)**
- 하는 일: "서울 날씨 어때?"라는 질문과 함께, 사용할 수 있는 툴 목록(JSON Schema)을 LLM에 보냅니다.

**2단계: LLM ➡ 애플리케이션 (판단 및 호출 요청)**
- 하는 일: LLM은 정보를 분석한 뒤, 본인이 직접 날씨를 확인하는 게 아니라 "내가 판단해보니 get_current_weather 함수를 location='서울'로 실행해야겠어. 나한테 그 결과값 좀 가져다줄래?" 하고 호출 문서(Tool Calls JSON)를 생성하여 응답합니다.

**3단계: 애플리케이션 (실제 함수 호출 주체 🌟)**
- 하는 일: 파이썬 코드가 LLM의 응답을 받아 tool_calls가 포함되어 있는지 확인합니다. 포함되어 있다면, 개발자가 작성한 실제 파이썬 함수를 호출(get_current_weather('서울'))하여 결과 데이터(JSON)를 얻어냅니다.

**4단계: 애플리케이션 ➡ LLM (결과 전달 및 최종 답변)**
- 하는 일: 애플리케이션이 함수 실행 결과(예: {"temp": 22})를 다시 LLM에게 전송합니다. LLM은 이 데이터를 바탕으로 사람이 읽을 수 있는 최종 답변("현재 서울의 날씨는 22도입니다.")을 만들어 사용자에게 전달합니다.

**LLM의 역할: "이 함수를 이런 값으로 실행해라" 하고 결정(Decision)하는 역할.**

**애플리케이션(코드)의 역할: LLM의 결정을 보고 파이썬 함수를 실제 실행(Execution)하는 역할.**

**📝 정리** — 에이전트 루프의 4단계를 다시 정리했다. 모델은 **무엇을 어떤 인자로 실행할지 결정**만 하고, 실제 **실행**은 내 파이썬 코드가 한다. 그래서 함수 도구(function tool)를 쓰면 `function_call`을 받아 실행하고 `function_call_output`을 돌려주는 루프를 내가 짜야 한다.

**Schema 작성 원칙**

- 좋은 Tool Schema는 “모델이 추측하지 않아도 되게” 만드는 게 핵심입니다.

```
{
    "type": "function",
    "name": "verb_noun",
    "description": (
        "One-sentence purpose. "
        "Use this when ... "
        "Do not use this when ..."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "arg1": {
                "type": "string",
                "description": "Exact meaning, expected format, and example."
            },
            "arg2": {
                "type": ["string", "null"],
                "description": "Optional value. Use null if not specified."
            }
        },
        "required": ["arg1", "arg2"],
        "additionalProperties": False
    }
}
```

**📝 정리** — 좋은 스키마의 핵심은 "모델이 추측하지 않아도 되게" 쓰는 것이다.
- `name`은 `동사_명사` 형태(예: `get_current_weather`).
- `description`에 목적 한 문장 + **언제 쓰는지 / 언제 쓰면 안 되는지**를 적는다. 모델은 코드를 못 보고 이 설명만 보고 판단한다.
- 인자마다 의미, 형식, 예시를 적는다(10/12 날씨 도구에서 `'영문 도시명 (예: Seoul)'`이라고 적었더니 한국어 질문에서도 `Seoul`로 넘긴 것과 같은 원리).
- `"strict": True`(구조화 출력 모드)를 켜면 모든 속성을 `required`에 넣고 `additionalProperties: False`를 줘야 한다. 그래서 선택 인자는 빼는 대신 `["string", "null"]`처럼 null을 허용하는 방식으로 표현한다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

**Open AI의 도구 종류**

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

- 빌트인 툴 종류
> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [7]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="2026년 10월 1일 발표된 중요한 AI 뉴스 알려줘",
    tools=[
        {"type": "web_search"}
    ]
)

print(response.output_text)

2026년 10월 1일에 발표된 주요 인공지능(AI) 관련 뉴스는 다음과 같습니다:

1. **미국 증시 상승과 AI 관련주 강세**: 미국 국채금리 급등에도 불구하고, 마이크론의 호실적과 알파벳의 새로운 AI 모델 공개로 인해 기술주 투자 심리가 개선되며 AI 관련 주식이 상승세를 보였습니다. ([newspim.com](https://www.newspim.com/news/view/20261001001431?utm_source=openai))

2. **'모두의 AI' 프로젝트 발표**: SK텔레콤, 카카오, KT는 10월 중에 전 국민이 무료로 이용할 수 있는 국산 AI 서비스 '모두의 AI'를 선보일 예정이라고 발표했습니다. 이 서비스는 전화, 문자, 카카오톡 등 일상적인 서비스에서 예약, 신청, 결제까지 지원하는 AI 에이전트를 제공하여 해외 AI 의존도를 낮추고 국민 간 AI 활용 격차를 해소하는 것을 목표로 합니다. ([yna.co.kr](https://www.yna.co.kr/amp/view/AKR20260930159700017?utm_source=openai))

3. **구글의 Gemini Live 'Guided Vision' 출시**: 구글은 호환되는 안드로이드 기기에서 Gemini Live의 'Guided Vision' 기능을 출시했습니다. 이 기능은 사용자가 주변 환경을 인식하고 이해하는 데 도움을 주는 시각 보조 도구로, AI 기술을 활용하여 실시간으로 정보를 제공합니다. ([blogformula.net](https://www.blogformula.net/ko/11421?utm_source=openai))

이러한 소식들은 AI 기술의 발전과 일상 생활에의 통합이 가속화되고 있음을 보여줍니다. 


**📝 정리** — `tools=[{"type": "web_search"}]` 한 줄로 **빌트인 툴**을 붙였다. 함수 도구와 달리 검색은 **OpenAI 서버가 직접 실행**하고 결과까지 반영해서 답을 주므로, 10/12처럼 `function_call`을 받아 실행하는 루프가 필요 없다. 모델이 모르는 날짜(2026년 10월 1일)의 뉴스를 출처 링크와 함께 답했다. 링크에 `utm_source=openai`가 붙은 것으로 웹 검색 결과를 인용했다는 걸 알 수 있다. (뉴스 내용 자체는 따로 확인하지 않았다)

**MCP Tool**

- Function tool은 “내 앱 코드 안에 직접 등록한 함수”이고, MCP tool은 “외부 MCP 서버가 제공하는 도구들을 표준 프로토콜로 연결한 것”입니다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [8]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="서울의 현재 기온, 습도를 알려줘.",
    tools=[
        {
            "type": "mcp",
            "server_label": "open_meteo",
            "server_url": "https://open-meteo.caseyjhand.com/mcp",
            "require_approval": "never",
            "allowed_tools": [
                "openmeteo_search_locations",
                "openmeteo_get_forecast",
            ],
        }
    ],
)

print(response.output_text)

현재 서울의 기온은 **16.5°C**이며, 습도는 **86%**입니다.


**📝 정리** — **MCP 툴**: 외부 MCP 서버(`open_meteo`, 날씨 서버)를 `server_url`로 연결했다. 빌트인 툴처럼 OpenAI 서버가 MCP 서버에 접속해 도구 목록을 받고 호출까지 하므로, 내 코드는 요청 한 번으로 끝난다.
- `allowed_tools`: 서버가 제공하는 도구 중 위치 검색(`openmeteo_search_locations`)과 예보 조회(`openmeteo_get_forecast`) 두 개만 쓰도록 제한했다. 질문의 "서울"을 좌표로 바꾸고 날씨를 조회하는 데 필요한 조합이다.
- `require_approval: "never"`: 도구를 호출할 때마다 승인을 받지 않는다. 편하지만 외부 서버를 그대로 믿는 설정이라, 내가 만든 서버나 믿을 수 있는 서버에만 쓰는 게 안전하다.
- 결과는 서울 기온 16.5°C, 습도 86%. 같은 날 아침 `wttr.in` 함수 도구로 조회한 값(17°C, 84%)과 비슷하지만, 조회한 시각과 데이터 출처가 달라서 숫자가 조금 다르다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

<hr style="background-color:red; height:3px">

### <span style='color:blue; font-weight:bold'>Tool(Function, Builtin, MCP)호출</span>에 대해 학습했습니다.

## 🔁 복습 질문

<details>
<summary>1. 빌트인 <code>web_search</code> 결과에서 무엇으로 출처를 확인하나?</summary>

응답 메시지의 주석(annotations)에 들어 있는 URL 인용.
</details>

<details>
<summary>2. MCP 툴을 쓸 때 승인(<code>require_approval</code>) 설정을 고민해야 하는 이유는?</summary>

외부 서버가 실제 동작을 할 수 있어서, 민감한 도구는 사람이 승인한 뒤 실행되게 해야 안전하다.
</details>
